In [1]:
import sys, os, random
sys.path.append(os.path.join(os.getcwd(), 'CONFOLD')) #add CONFOLD to path

import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import pandas as pd

from foldrm import Classifier
from datasets import  final_extinctionrisk_dataframe, final_extinctionrisk_noth_dataframe

from algo import prune_rules

In [2]:
random.seed(42)

# Without Human Threats

In [3]:
model_template, data = final_extinctionrisk_noth_dataframe()
X = data[model_template.attrs].drop('Extinction_risk',axis=1)
X = X.convert_dtypes()
X = X.to_numpy()
y = data['Extinction_risk'].to_numpy()

# Loading the Train/Test Splits from R

In [4]:
import csv

train_indices = []
test_indices = []
for i in range(1,11):
    with open("./datasets/r_folds/fold_"+str(i)+"_train_idx.csv", newline='') as csvfile:
        train_df = pd.read_csv(csvfile)
        train_list = train_df.to_numpy().flatten().tolist()
        train_indices.append(train_list)
    with open("./datasets/r_folds/fold_"+str(i)+"_test_idx.csv", newline='') as csvfile:
        test_df = pd.read_csv(csvfile)
        test_list = test_df.to_numpy().flatten().tolist()
        test_indices.append(test_list)

# K-Fold Cross Validation Without Human Threats.
- Data Driven Model

In [5]:
acc_metrics = []
f1_metrics = [] 
precision_metrics = []
recall_metrics = []


for i, (train_index, test_index) in enumerate(zip(train_indices, test_indices)):
    print("Fold "+str(i)+":")
    
    train_data = np.concatenate((np.array(X[train_index]), np.array(y[train_index]).reshape(-1, 1)), axis=1).tolist()
    test_data = np.concatenate((np.array(X[test_index]), np.array(y[test_index]).reshape(-1, 1)), axis=1).tolist()
 
    # Prepare the test data (features and true labels)
    X_test = [d[:-1] for d in test_data]
    Y_test = [d[-1] for d in test_data]

    advanced_pruning_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)

    # Now, train using confidence_fit with a high 15% improvement threshold
    advanced_pruning_model.confidence_fit(train_data, improvement_threshold=0.20)
    advanced_pruning_model.print_asp(simple=True)


    adv_pruning_predictions = advanced_pruning_model.predict(X_test)
    adv_pruning_labels = [p[0] for p in adv_pruning_predictions]

    for i in range(len(adv_pruning_labels)):
        if adv_pruning_labels[i] is None:
            adv_pruning_labels[i] = ('None')
    acc = accuracy_score(Y_test, adv_pruning_labels)

    f1 = f1_score(Y_test, adv_pruning_labels, average=None, labels=["Higher_risk", "Lower_risk"]) #["Higher_risk", "Lower_risk"]["Lower_risk", "Higher_risk"]
    precision = precision_score(Y_test,adv_pruning_labels, average=None, labels=["Higher_risk", "Lower_risk"])
    recall = recall_score(Y_test, adv_pruning_labels, average=None, labels=["Higher_risk", "Lower_risk"])
    
    acc_metrics.append(acc)    
    f1_metrics.append(f1)
    precision_metrics.append(precision)
    recall_metrics.append(recall)

    print("----------------------------------------")
    print()

Fold 0:
extinction_risk(X,'lower_risk') :- range_size(X,N21), N21>15797.46. [confidence: 0.87839]
extinction_risk(X,'higher_risk') :- range_size(X,N21), N21<=3270.79. [confidence: 0.75422]
extinction_risk(X,'lower_risk') :- beak_depth(X,N3), N3<=12.1, minimum_latitude(X,N8), N8>-21.91, island_restricted_breeding(X,N11), N11>0. [confidence: 0.69669]
extinction_risk(X,'higher_risk') :- generation_length(X,N20), N20>3.6433. [confidence: 0.77542]
extinction_risk(X,'lower_risk') :- minimum_latitude(X,N8), N8<=9.98, maximum_latitude(X,N9), N9>-0.17. [confidence: 0.77326]
extinction_risk(X,'higher_risk') :- wing_length(X,N5), N5>46.7. [confidence: 0.70763]
extinction_risk(X,'lower_risk') :- not order(X,'caprimulgiformes'). [confidence: 0.65385]
extinction_risk(X,'lower_risk') :- order(X,'caprimulgiformes'). [confidence: 0.55]
----------------------------------------

Fold 1:
extinction_risk(X,'lower_risk') :- range_size(X,N21), N21>15801.31. [confidence: 0.87671]
extinction_risk(X,'higher_ris

In [6]:
print("Data Driven Model")
print(f"KFold Mean Acc: {round(np.mean(acc_metrics), 5)} | KFold SD: {np.std(acc_metrics)}")
f1_metrics = np.array(f1_metrics)
precision_metrics = np.array(precision_metrics)
recall_metrics = np.array(recall_metrics)

print(f"KFold Mean Precision: Higher Risk {round(np.mean(precision_metrics[:,0]), 5)}, Lower Risk {round(np.mean(precision_metrics[:,1]), 5)}  | KFold SD: {round(np.std(precision_metrics[:, 0]),5), round(np.std(precision_metrics[:, 1]), 5)}")
print(f"KFold Mean Recall: Higher Risk {round(np.mean(recall_metrics[:,0]), 5)}, Lower Risk {round(np.mean(recall_metrics[:,1]), 5)} | KFold SD: {round(np.std(recall_metrics[:, 0]), 5), round(np.std(recall_metrics[:, 1]), 5)}")
print(f"KFold Mean f1: Higher Risk {round(np.mean(f1_metrics[:,0]), 5)}, Lower Risk {round(np.mean(f1_metrics[:,1]), 5)} | KFold SD: {round(np.std(f1_metrics[:,0]), 5), round(np.std(f1_metrics[:,1]), 5)}")

Data Driven Model
KFold Mean Acc: 0.8511 | KFold SD: 0.009967351334719089
KFold Mean Precision: Higher Risk 0.7081, Lower Risk 0.86682  | KFold SD: (np.float64(0.0433), np.float64(0.00803))
KFold Mean Recall: Higher Risk 0.36825, Lower Risk 0.96453 | KFold SD: (np.float64(0.04266), np.float64(0.00518))
KFold Mean f1: Higher Risk 0.48375, Lower Risk 0.91305 | KFold SD: (np.float64(0.04501), np.float64(0.00555))


# K-Fold Cross Validation Without Human Threats.
- Expert Rule Model

In [7]:
acc_metrics = []
f1_metrics = [] 
precision_metrics = []
recall_metrics = []


for i, (train_index, test_index) in enumerate(zip(train_indices, test_indices)):
    print("Fold "+str(i)+":")
    
    train_data = np.concatenate((np.array(X[train_index]), np.array(y[train_index]).reshape(-1, 1)), axis=1).tolist()
    test_data = np.concatenate((np.array(X[test_index]), np.array(y[test_index]).reshape(-1, 1)), axis=1).tolist()
 
    # Prepare the test data (features and true labels)
    X_test = [d[:-1] for d in test_data]
    Y_test = [d[-1] for d in test_data]

    # Instantiate a new classifier for expert-guided model
    expert_model = Classifier(attrs=model_template.attrs.copy(), numeric=model_template.numeric, label=model_template.label)

    # Define our expert rules as strings
    rule1 = "with confidence 0.99 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Body_mass' '>=' '130' and 'Generation_length' '>=' '4.068' and 'Elevational_range' '<=' '800'"
    rule2 = "with confidence 0.95 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Body_mass' '>=' '130' and 'Elevational_range' '<=' '800'"
    rule3 = "with confidence 0.95 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Generation_length' '>=' '4.068' and 'Elevational_range' '<=' '800'"
    rule4 = "with confidence 0.95 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Body_mass' '>=' '130' and 'Generation_length' '>=' '4.068'"
    rule5 = "with confidence 0.95 class = 'Higher_risk' if 'Body_mass' '>=' '130' and 'Generation_length' '>=' '4.068' and 'Elevational_range' '<=' '800'"

    # Add the manual rules to the model
    expert_model.add_manual_rule(rule1, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
    expert_model.add_manual_rule(rule2, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
    expert_model.add_manual_rule(rule3, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
    expert_model.add_manual_rule(rule4, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
    expert_model.add_manual_rule(rule5, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)

    #expert_model.fit(train_data, ratio=0.75)

    expert_model.confidence_fit(train_data, improvement_threshold=0.20)
    expert_model.print_asp(simple=True)


    expert_predictions_tuples = expert_model.predict(X_test)
    expert_predicted_labels = [p[0] for p in expert_predictions_tuples]

    acc = accuracy_score(Y_test, expert_predicted_labels)

    f1 = f1_score(Y_test, expert_predicted_labels, average=None, labels=["Higher_risk", "Lower_risk"]) #["Higher_risk", "Lower_risk"]["Lower_risk", "Higher_risk"]
    precision = precision_score(Y_test,expert_predicted_labels, average=None, labels=["Higher_risk", "Lower_risk"])
    recall = recall_score(Y_test, expert_predicted_labels, average=None, labels=["Higher_risk", "Lower_risk"])
    
    acc_metrics.append(acc)    
    f1_metrics.append(f1)
    precision_metrics.append(precision)
    recall_metrics.append(recall)

    print("----------------------------------------")
    print()

Fold 0:
extinction_risk(X,'higher_risk') :- elevational_range(X,N13), N13<=800, generation_length(X,N20), N20>4.068, range_size(X,N21), N21<=75321, body_mass(X,N22), N22>130. [confidence: 0.99]
extinction_risk(X,'higher_risk') :- elevational_range(X,N13), N13<=800, range_size(X,N21), N21<=75321, body_mass(X,N22), N22>130. [confidence: 0.95]
extinction_risk(X,'higher_risk') :- elevational_range(X,N13), N13<=800, generation_length(X,N20), N20>4.068, range_size(X,N21), N21<=75321. [confidence: 0.95]
extinction_risk(X,'higher_risk') :- generation_length(X,N20), N20>4.068, range_size(X,N21), N21<=75321, body_mass(X,N22), N22>130. [confidence: 0.95]
extinction_risk(X,'higher_risk') :- elevational_range(X,N13), N13<=800, generation_length(X,N20), N20>4.068, body_mass(X,N22), N22>130. [confidence: 0.95]
extinction_risk(X,'lower_risk') :- range_size(X,N21), N21>14884.37. [confidence: 0.90306]
extinction_risk(X,'higher_risk') :- range_size(X,N21), N21<=3270.79. [confidence: 0.72162]
extinction_r

In [8]:
print("Expert Rule Model")
print(f"KFold Mean Acc: {round(np.mean(acc_metrics), 5)} | KFold SD: {np.std(acc_metrics)}")
f1_metrics = np.array(f1_metrics)
precision_metrics = np.array(precision_metrics)
recall_metrics = np.array(recall_metrics)

print(f"KFold Mean Precision: Higher Risk {round(np.mean(precision_metrics[:,0]), 5)}, Lower Risk {round(np.mean(precision_metrics[:,1]), 5)}  | KFold SD: {round(np.std(precision_metrics[:, 0]),5), round(np.std(precision_metrics[:, 1]), 5)}")
print(f"KFold Mean Recall: Higher Risk {round(np.mean(recall_metrics[:,0]), 5)}, Lower Risk {round(np.mean(recall_metrics[:,1]), 5)} | KFold SD: {round(np.std(recall_metrics[:, 0]), 5), round(np.std(recall_metrics[:, 1]), 5)}")
print(f"KFold Mean f1: Higher Risk {round(np.mean(f1_metrics[:,0]), 5)}, Lower Risk {round(np.mean(f1_metrics[:,1]), 5)} | KFold SD: {round(np.std(f1_metrics[:,0]), 5), round(np.std(f1_metrics[:,1]), 5)}")

Expert Rule Model
KFold Mean Acc: 0.83011 | KFold SD: 0.01445848427818524
KFold Mean Precision: Higher Risk 0.55808, Lower Risk 0.88908  | KFold SD: (np.float64(0.04093), np.float64(0.01119))
KFold Mean Recall: Higher Risk 0.51982, Lower Risk 0.90301 | KFold SD: (np.float64(0.0534), np.float64(0.01348))
KFold Mean f1: Higher Risk 0.53723, Lower Risk 0.89591 | KFold SD: (np.float64(0.04186), np.float64(0.00901))


# Training on Complete Dataset 
- withought Expert Rules 
- and without Human Threats

In [9]:
baseline_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
train_data = np.concatenate((np.array(X), np.array(y).reshape(-1, 1)), axis=1).tolist()

advanced_pruning_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
advanced_pruning_model.confidence_fit(train_data, improvement_threshold=0.20)
advanced_pruning_model.print_asp(simple=True)

extinction_risk(X,'lower_risk') :- range_size(X,N21), N21>15801.31. [confidence: 0.8779]
extinction_risk(X,'higher_risk') :- range_size(X,N21), N21<=3270.79. [confidence: 0.75379]
extinction_risk(X,'lower_risk') :- beak_depth(X,N3), N3<=12.1, minimum_latitude(X,N8), N8>-21.91, island_restricted_breeding(X,N11), N11>0. [confidence: 0.69732]
extinction_risk(X,'higher_risk') :- maximum_elevation(X,N18), N18<=2750.0. [confidence: 0.67375]
extinction_risk(X,'lower_risk') :- beak_length_culmen(X,N2), N2<=18.8. [confidence: 0.78814]
extinction_risk(X,'higher_risk') :- minimum_latitude(X,N8), N8>8.76. [confidence: 0.73529]
extinction_risk(X,'lower_risk') :- tail_length(X,N7), N7>95.3. [confidence: 0.65385]
extinction_risk(X,'higher_risk') :- wing_length(X,N5), N5>91.3. [confidence: 0.67857]
extinction_risk(X,'lower_risk') :- range_size(X,N21), N21>7907.78. [confidence: 0.61765]
extinction_risk(X,'higher_risk') :- order(X,'caprimulgiformes'). [confidence: 0.59091]
extinction_risk(X,'higher_risk

# Training on Complete Dataset 
- With Expert Rules 
- and without Human Threats

In [10]:
train_data = np.concatenate((np.array(X), np.array(y).reshape(-1, 1)), axis=1).tolist()

expert_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)

# Define our expert rules as strings
rule1 = "with confidence 0.99 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Body_mass' '>=' '130' and 'Generation_length' '>=' '4.068' and 'Elevational_range' '<=' '800'"
rule2 = "with confidence 0.95 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Body_mass' '>=' '130' and 'Elevational_range' '<=' '800'"
rule3 = "with confidence 0.95 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Generation_length' '>=' '4.068' and 'Elevational_range' '<=' '800'"
rule4 = "with confidence 0.95 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Body_mass' '>=' '130' and 'Generation_length' '>=' '4.068'"
rule5 = "with confidence 0.95 class = 'Higher_risk' if 'Body_mass' '>=' '130' and 'Generation_length' '>=' '4.068' and 'Elevational_range' '<=' '800'"

# Add the manual rules to the model
expert_model.add_manual_rule(rule1, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
expert_model.add_manual_rule(rule2, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
expert_model.add_manual_rule(rule3, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
expert_model.add_manual_rule(rule4, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
expert_model.add_manual_rule(rule5, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)

#expert_model.fit(train_data, ratio=0.75)

expert_model.confidence_fit(train_data, improvement_threshold=0.20)
expert_model.print_asp(simple=True)

extinction_risk(X,'higher_risk') :- elevational_range(X,N13), N13<=800, generation_length(X,N20), N20>4.068, range_size(X,N21), N21<=75321, body_mass(X,N22), N22>130. [confidence: 0.99]
extinction_risk(X,'higher_risk') :- elevational_range(X,N13), N13<=800, range_size(X,N21), N21<=75321, body_mass(X,N22), N22>130. [confidence: 0.95]
extinction_risk(X,'higher_risk') :- elevational_range(X,N13), N13<=800, generation_length(X,N20), N20>4.068, range_size(X,N21), N21<=75321. [confidence: 0.95]
extinction_risk(X,'higher_risk') :- generation_length(X,N20), N20>4.068, range_size(X,N21), N21<=75321, body_mass(X,N22), N22>130. [confidence: 0.95]
extinction_risk(X,'higher_risk') :- elevational_range(X,N13), N13<=800, generation_length(X,N20), N20>4.068, body_mass(X,N22), N22>130. [confidence: 0.95]
extinction_risk(X,'lower_risk') :- range_size(X,N21), N21>16469.54. [confidence: 0.90208]
extinction_risk(X,'higher_risk') :- range_size(X,N21), N21<=3270.79. [confidence: 0.72233]
extinction_risk(X,'l